# Notebook 6 — Checkpoint Evaluation & Results Table

Loads each saved checkpoint and evaluates it over 20 held-out episodes. Produces a summary table and learning curve plot.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/RL_Models'
except ImportError:
    SAVE_DIR = 'models'

import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gymnasium as gym
import ale_py
from stable_baselines3 import DQN, A2C, PPO
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack
from stable_baselines3.common.evaluation import evaluate_policy

gym.register_envs(ale_py)

ALGO_MAP = {'dqn': DQN, 'a2c': A2C, 'ppo': PPO}
N_EVAL_EPISODES = 20

In [ ]:
def evaluate_checkpoint(algo_name, checkpoint_path, env_id, n_episodes=20):
    AlgoClass = ALGO_MAP[algo_name]
    eval_env = make_atari_env(env_id, n_envs=1, seed=999)
    eval_env = VecFrameStack(eval_env, n_stack=4)
    model = AlgoClass.load(checkpoint_path, env=eval_env)
    mean_r, std_r = evaluate_policy(model, eval_env, n_eval_episodes=n_episodes, deterministic=True)
    eval_env.close()
    return mean_r, std_r

In [ ]:
# Define which runs to evaluate
EVAL_RUNS = [
    {'algo': 'dqn', 'run_dir': os.path.join(SAVE_DIR, 'dqn_pong_default'), 'env': 'ALE/Pong-v5'},
    # Add more runs as they complete:
    # {'algo': 'a2c', 'run_dir': os.path.join(SAVE_DIR, 'a2c_default_Pong_v5'), 'env': 'ALE/Pong-v5'},
    # {'algo': 'ppo', 'run_dir': os.path.join(SAVE_DIR, 'ppo_default_Pong_v5'), 'env': 'ALE/Pong-v5'},
]

records = []
for run in EVAL_RUNS:
    ckpts = sorted(glob.glob(os.path.join(run['run_dir'], f"{run['algo']}_*_steps.zip")))
    for ckpt in ckpts:
        # Extract timestep from filename
        steps = int(os.path.basename(ckpt).split('_steps')[0].split('_')[-1])
        print(f"Evaluating {run['algo'].upper()} @ {steps:,} steps...")
        mean_r, std_r = evaluate_checkpoint(run['algo'], ckpt, run['env'], N_EVAL_EPISODES)
        records.append({
            'Algorithm': run['algo'].upper(),
            'Environment': run['env'],
            'Timesteps': steps,
            'Mean Reward': round(mean_r, 2),
            'Std Reward': round(std_r, 2),
        })

df = pd.DataFrame(records).sort_values(['Algorithm', 'Timesteps'])
print(df.to_string(index=False))

In [ ]:
# Learning curve plot
fig, ax = plt.subplots(figsize=(10, 5))
for algo, group in df.groupby('Algorithm'):
    ax.plot(group['Timesteps'], group['Mean Reward'], marker='o', label=algo)
    ax.fill_between(
        group['Timesteps'],
        group['Mean Reward'] - group['Std Reward'],
        group['Mean Reward'] + group['Std Reward'],
        alpha=0.2
    )
ax.set_xlabel('Training Timesteps')
ax.set_ylabel('Mean Reward (20 episodes)')
ax.set_title('Algorithm Comparison on Pong')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'learning_curves.png'), dpi=150)
plt.show()